# Binary `isBird` classifier from the bridge and ESP32 manifests

This notebook trains and exports a binary bird/not-bird TensorFlow Lite model using exactly these dataset authorities:

- `E:\Datasets\ESP32_capture\manifests\esp32_capture_manifest_generated.csv`
  - target-camera bird and not-bird examples;
  - authoritative binary label: `bird_present`;
  - every accepted row must have a matching, readable capture sidecar containing a valid acquisition-time ROI at `camera.requested.motion.roi`.
- `C:\Users\Joe\Desktop\ESP_Classifier\esp32_Classifier\Notebooks\Outputs\Bridge Outputs\bird_bridge_manifest.csv`
  - CUB-200-2011 and iNaturalist bird-positive augmentation;
  - rows must be valid Aves images;
  - bridge inputs deliberately retain the whole frame because they have no ESP32 capture ROI provenance.

## Design

- **Labels:** `0 = not_bird`, `1 = bird`.
- **Input:** RGB `224 × 224`. ESP32 images are cropped to their recorded normalized ROI with a 20%-total-dimension padding (10% per edge before clipping), then letterboxed. Bridge images are letterboxed whole-frame. Pixel scaling from `[0, 255]` to `[-1, 1]` is embedded in the model.
- **Strict provenance:** an ESP32 row is excluded—not silently treated as whole-frame—when its matching sidecar or recorded ROI is missing, mismatched, unreadable, or invalid. Rejection reasons are displayed and saved with the run.
- **Leakage control:** ESP32 validation and test sets are held out by complete capture date. Bridge images are training-only, so the primary metrics remain target-camera metrics.
- **Balanced training stream:** 50% ESP32 negatives, 30% ESP32 birds, and 20% bridge birds by default. This prevents 86k bridge positives from swamping the limited camera negatives.
- **Threshold:** selected only from **uint8 TFLite validation predictions**, maximizing ESP32 bird recall subject to the configured ESP32 false-positive-rate limit.
- **Exports:** Keras, float32 TFLite, and full-integer uint8 TFLite, plus dataset, provenance-rejection, threshold, metrics, and inference-contract artifacts.

Run the notebook from top to bottom. Set `QUICK_RUN = True` only for a short pipeline check; its metrics are not suitable for model selection.


## 1. Configuration


In [1]:
# ---- Input manifests ---------------------------------------------------------
ESP32_MANIFEST = r"E:\Datasets\ESP32_capture\manifests\esp32_capture_manifest_generated.csv"
BRIDGE_MANIFEST = r"C:\Users\Joe\Desktop\ESP_Classifier\esp32_Classifier\Notebooks\Outputs\Bridge Outputs\bird_bridge_manifest.csv"
ESP32_CAPTURE_ROOT = r"E:\Datasets\ESP32_capture"

# Whole dates are held out to prevent near-duplicate burst frames from leaking
# across splits. Unconfigured dates must precede every configured holdout date.
ESP32_VALIDATION_DATES = ("2026-10-12", "2026-10-13")
ESP32_TEST_DATES = ("2026-10-14",)

# ---- Image/model -------------------------------------------------------------
PREPROCESSING = "letterbox"  # ROI crops and whole bridge frames both use resize_with_pad.
IMG_SIZE = 224
ESP32_CROP_SOURCE = "capture_motion_record.selection.final_bounds_px"
ROI_COORDINATE_SPACE = "normalized_original_frame"
ROI_PADDING_FRACTION = 0.20  # Total width/height increase: 10% on each edge before clipping.
ROI_PADDING_SEMANTICS = "total_dimension_increase"
ROI_PIXEL_ROUNDING = "floor_min_ceil_max"
ESP32_CROP_POLICY_NAME = "pi_selected_crop_then_letterbox"
BRIDGE_CROP_POLICY_NAME = "whole_frame_then_letterbox"
BACKBONE_ALPHA = 1.0         # MobileNetV2: 0.35 / 0.5 / 0.75 / 1.0
WEIGHTS = "imagenet"
DROPOUT = 0.2

# ---- Training ---------------------------------------------------------------
BATCH_SIZE = 32
TRAIN_EXAMPLES_PER_EPOCH = 12_000
TRAIN_SOURCE_WEIGHTS = {
    "esp32_not_bird": 0.50,
    "esp32_bird": 0.30,
    "bridge_bird": 0.20,
}
HEAD_EPOCHS = 10
FINE_TUNE_EPOCHS = 10
FINE_TUNE_LAYERS = 40
HEAD_LR = 1e-3
FINE_TUNE_LR = 1e-5
EARLY_STOP_PATIENCE = 4
AUGMENT_FLIP = True
AUGMENT_COLOR = True
SEED = 1234

# ---- Validation/export -------------------------------------------------------
VERIFY_ALL_FILES = True
MAX_ESP32_VALIDATION_FALSE_POSITIVE_RATE = 0.05
REPRESENTATIVE_IMAGES = 300
TFLITE_EVAL_LIMIT = None  # None evaluates every validation/test image.
RUN_NAME = None           # default: isbird_manifests_<geometry>_<timestamp>

# A fast end-to-end check. Do not compare its metrics with a full run.
QUICK_RUN = False
if QUICK_RUN:
    VERIFY_ALL_FILES = False
    TRAIN_EXAMPLES_PER_EPOCH = 512
    HEAD_EPOCHS = 1
    FINE_TUNE_EPOCHS = 1
    EARLY_STOP_PATIENCE = 1
    REPRESENTATIVE_IMAGES = 30
    TFLITE_EVAL_LIMIT = 128

assert PREPROCESSING == "letterbox"
assert 0.0 <= ROI_PADDING_FRACTION < 1.0
assert set(TRAIN_SOURCE_WEIGHTS) == {"esp32_not_bird", "esp32_bird", "bridge_bird"}
assert abs(sum(TRAIN_SOURCE_WEIGHTS.values()) - 1.0) < 1e-9
assert not (set(ESP32_VALIDATION_DATES) & set(ESP32_TEST_DATES))


## 2. Environment and portable path helpers


In [2]:
import hashlib
import json
import os
import platform
import random
import re
import time
from datetime import datetime, timezone
from decimal import Decimal
from pathlib import Path, PurePosixPath, PureWindowsPath

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

keras = tf.keras

for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


def find_models_dir():
    """Find the repository Models directory without assuming notebook CWD."""
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "Models", cwd.parent / "Models"]
    for parent in (cwd, *cwd.parents):
        candidates.extend((parent / "Models", parent))
    for candidate in candidates:
        if candidate.name == "Models" and (candidate / "Training Manifest").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(f"Could not find the Models directory from {cwd}")


def portable_path(raw_path):
    """Map Windows paths to WSL and /mnt/<drive> paths to Windows."""
    raw = str(raw_path).strip()
    windows_path = PureWindowsPath(raw)
    if windows_path.drive:
        if os.name == "nt":
            return Path(raw)
        drive = windows_path.drive.rstrip(":").lower()
        return Path("/mnt") / drive / Path(*windows_path.parts[1:])
    mount_match = re.match(r"^/mnt/([A-Za-z])/(.*)$", raw.replace("\\", "/"))
    if mount_match:
        if os.name != "nt":
            return Path(raw)
        drive, rest = mount_match.groups()
        return Path(f"{drive.upper()}:/") / Path(*PurePosixPath(rest).parts)
    return Path(raw)


def relative_parts(raw_path):
    return PurePosixPath(str(raw_path).replace("\\", "/").lstrip("/")).parts


def resolve_image_path(stored_path, relative_path=None, fallback_root=None):
    candidates = []
    if pd.notna(stored_path) and str(stored_path).strip():
        candidates.append(portable_path(stored_path))
    if fallback_root is not None and pd.notna(relative_path) and str(relative_path).strip():
        candidates.append(Path(fallback_root).joinpath(*relative_parts(relative_path)))
    if len(candidates) == 1:
        return str(candidates[0])
    for candidate in candidates:
        if candidate.is_file():
            return str(candidate)
    return str(candidates[-1]) if candidates else ""


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def strict_bool(series, column_name):
    true_values = {True, 1, 1.0, "1", "true", "yes", "y"}
    false_values = {False, 0, 0.0, "0", "false", "no", "n"}

    def parse(value):
        if pd.isna(value):
            return pd.NA
        normalized = value.strip().lower() if isinstance(value, str) else value
        if normalized in true_values:
            return True
        if normalized in false_values:
            return False
        raise ValueError(f"Unexpected boolean value {value!r} in {column_name}")

    return series.map(parse).astype("boolean")


def row_value_is_blank(value):
    return value is None or (isinstance(value, float) and np.isnan(value)) or not str(value).strip()


def normalized_stem(value):
    if row_value_is_blank(value):
        return ""
    return PurePosixPath(str(value).replace("\\", "/")).stem


def extract_recorded_crop(row):
    """Strictly read final Pi-selected pixel bounds from matching motion provenance."""
    result = {
        "crop_valid": False,
        "crop_rejection_reason": "",
        "crop_left": np.nan, "crop_top": np.nan, "crop_right": np.nan, "crop_bottom": np.nan,
        "crop_source": ESP32_CROP_SOURCE,
        "motion_record_path": "", "motion_record_sha256": "", "motion_config_sha256": "",
    }
    image_path = resolve_image_path(row.get("image_path"), row.get("image_relative_path"), ESP32_CAPTURE_ROOT_PATH)
    if not image_path:
        result["crop_rejection_reason"] = "image_path_missing"
        return result
    motion_path = str(Path(image_path).with_suffix(".motion.json"))
    result["motion_record_path"] = motion_path
    path = Path(motion_path)
    if not path.is_file():
        result["crop_rejection_reason"] = "motion_record_missing"
        return result
    try:
        raw = path.read_bytes()
        motion = json.loads(raw.decode("utf-8"))
    except (OSError, UnicodeDecodeError, json.JSONDecodeError):
        result["crop_rejection_reason"] = "motion_record_invalid"
        return result
    result["motion_record_sha256"] = hashlib.sha256(raw).hexdigest()
    capture = motion.get("capture") if isinstance(motion, dict) else None
    selection = motion.get("selection") if isinstance(motion, dict) else None
    producer = motion.get("producer") if isinstance(motion, dict) else None
    if not isinstance(capture, dict) or capture.get("capture_id") != row.get("record_id"):
        result["crop_rejection_reason"] = "motion_capture_id_mismatch"
        return result
    if capture.get("image_sha256") != row.get("image_sha256"):
        result["crop_rejection_reason"] = "motion_image_sha256_mismatch"
        return result
    static_roi = motion.get("static_roi") if isinstance(motion, dict) else None
    sidecar_path = resolve_image_path(
        row.get("sidecar_path"), row.get("sidecar_relative_path"), ESP32_CAPTURE_ROOT_PATH
    )
    if not sidecar_path or not Path(sidecar_path).is_file():
        result["crop_rejection_reason"] = "capture_sidecar_missing"
        return result
    if not isinstance(static_roi, dict) or static_roi.get("capture_sidecar") != Path(sidecar_path).name:
        result["crop_rejection_reason"] = "motion_sidecar_name_mismatch"
        return result
    if static_roi.get("capture_sidecar_sha256") != sha256_file(sidecar_path):
        result["crop_rejection_reason"] = "motion_sidecar_sha256_mismatch"
        return result
    if not isinstance(selection, dict) or selection.get("coordinate_space") != "original_decoded_frame_pixels" or selection.get("bounds_semantics") != "half_open_left_top_right_bottom":
        result["crop_rejection_reason"] = "motion_selection_contract_invalid"
        return result
    bounds = selection.get("final_bounds_px")
    keys = ("left", "top", "right", "bottom")
    if not isinstance(bounds, dict) or any(isinstance(bounds.get(k), bool) or not isinstance(bounds.get(k), int) for k in keys):
        result["crop_rejection_reason"] = "motion_bounds_invalid"
        return result
    left, top, right, bottom = (bounds[k] for k in keys)
    width, height = int(capture.get("width", 0)), int(capture.get("height", 0))
    if not (0 <= left < right <= width and 0 <= top < bottom <= height):
        result["crop_rejection_reason"] = "motion_bounds_outside_frame"
        return result
    result.update({
        "crop_valid": True, "crop_left": left, "crop_top": top,
        "crop_right": right, "crop_bottom": bottom,
        "motion_config_sha256": (producer or {}).get("config_sha256", ""),
    })
    return result


MODELS_DIR = find_models_dir()
RUNS_DIR = MODELS_DIR / "runs"
ESP32_MANIFEST_PATH = portable_path(ESP32_MANIFEST)
BRIDGE_MANIFEST_PATH = portable_path(BRIDGE_MANIFEST)
ESP32_CAPTURE_ROOT_PATH = portable_path(ESP32_CAPTURE_ROOT)

for name, path in {
    "ESP32 manifest": ESP32_MANIFEST_PATH,
    "bridge manifest": BRIDGE_MANIFEST_PATH,
    "ESP32 capture root": ESP32_CAPTURE_ROOT_PATH,
}.items():
    if not path.exists():
        raise FileNotFoundError(f"{name} not found: {path}")

print("Python", platform.python_version(), "| TensorFlow", tf.__version__)
print("OS:", platform.platform())
print("GPUs:", tf.config.list_physical_devices("GPU"))
print("Models directory:", MODELS_DIR)
print("ESP32 manifest:", ESP32_MANIFEST_PATH)
print("Bridge manifest:", BRIDGE_MANIFEST_PATH)


I0000 00:00:1792973114.322246     969 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792973114.857951     969 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792973116.777846     969 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Python 3.12.15 | TensorFlow 2.21.0
OS: Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.43
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Models directory: /mnt/c/Users/Joe/Desktop/ESP_Classifier/esp32_Classifier/Models
ESP32 manifest: /mnt/e/Datasets/ESP32_capture/manifests/esp32_capture_manifest_generated.csv
Bridge manifest: /mnt/c/Users/Joe/Desktop/ESP_Classifier/esp32_Classifier/Notebooks/Outputs/Bridge Outputs/bird_bridge_manifest.csv


## 3. Load and validate both manifests

The notebook fails early on schema, label, split, duplicate, or file-integrity problems. In addition, it reads the **matching sidecar for every ESP32 row** and accepts only a finite, positive normalized rectangle at the immutable acquisition path `camera.requested.motion.roi`. It never substitutes the current camera configuration or a full-frame default. Missing and invalid provenance is excluded with a stable reason, summarized below, and written to the run's rejection report. In quick mode image-file verification checks deterministic samples instead of every accepted image.


In [3]:
ESP32_REQUIRED_COLUMNS = {
    "record_id", "image_relative_path", "sidecar_relative_path", "image_path", "sidecar_path",
    "pair_status", "image_sha256", "json_valid", "image_valid", "bird_present",
}
BRIDGE_REQUIRED_COLUMNS = {
    "bridge_image_id", "source_dataset", "absolute_path", "file_exists",
    "taxonomic_class", "width_px", "height_px",
}

esp32_raw = pd.read_csv(ESP32_MANIFEST_PATH, low_memory=False)
bridge_raw = pd.read_csv(BRIDGE_MANIFEST_PATH, low_memory=False)

missing = ESP32_REQUIRED_COLUMNS - set(esp32_raw.columns)
if missing:
    raise ValueError(f"ESP32 manifest is missing columns: {sorted(missing)}")
missing = BRIDGE_REQUIRED_COLUMNS - set(bridge_raw.columns)
if missing:
    raise ValueError(f"Bridge manifest is missing columns: {sorted(missing)}")

esp32 = esp32_raw.copy()
for column in ("json_valid", "image_valid", "bird_present"):
    esp32[column] = strict_bool(esp32[column], column)

# Read the per-capture sidecar itself. Manifest-level json_valid is necessary but
# cannot replace acquisition-provenance extraction from the matching JSON file.
crop_records = pd.DataFrame(
    [extract_recorded_crop(row) for _, row in esp32.iterrows()], index=esp32.index
)
for column in crop_records.columns:
    esp32[column] = crop_records[column]


def esp32_rejection_reasons(row):
    reasons = []
    if pd.isna(row["json_valid"]) or not bool(row["json_valid"]):
        reasons.append("manifest_json_invalid")
    if pd.isna(row["image_valid"]) or not bool(row["image_valid"]):
        reasons.append("manifest_image_invalid")
    if pd.isna(row["bird_present"]):
        reasons.append("label_missing")
    if str(row["pair_status"]).strip().lower() != "paired":
        reasons.append("pair_status_not_paired")
    if not bool(row["crop_valid"]):
        reasons.append(str(row["crop_rejection_reason"]) or "crop_invalid_unknown")
    return ";".join(reasons)


esp32["rejection_reasons"] = esp32.apply(esp32_rejection_reasons, axis=1)
valid_esp32 = esp32["rejection_reasons"].eq("")
rejected_esp32 = esp32.loc[~valid_esp32].copy()
esp32 = esp32.loc[valid_esp32].copy()

rejected_reason_counts = (
    rejected_esp32["rejection_reasons"].str.split(";").explode().value_counts().sort_index()
)
if esp32.empty:
    raise ValueError(
        "No ESP32 rows have valid matching Pi motion-crop provenance. Rejection counts: "
        f"{rejected_reason_counts.to_dict()}"
    )
if esp32["record_id"].duplicated().any():
    raise ValueError("ESP32 record_id values are not unique")
if esp32["image_sha256"].duplicated().any():
    raise ValueError("ESP32 image_sha256 values are not unique")

capture_stamp = esp32["record_id"].str.extract(
    r"^capture_(\d{8}T\d{6}_\d{6}Z)_", expand=False
)
if capture_stamp.isna().any():
    examples = esp32.loc[capture_stamp.isna(), "record_id"].head().tolist()
    raise ValueError(f"Could not parse capture time from record IDs: {examples}")
esp32["capture_datetime"] = pd.to_datetime(
    capture_stamp, format="%Y%m%dT%H%M%S_%fZ", utc=True, errors="raise"
)
esp32["capture_date"] = esp32["capture_datetime"].dt.strftime("%Y-%m-%d")
esp32["label"] = esp32["bird_present"].astype("int8")
esp32["class_name"] = esp32["label"].map({0: "not_bird", 1: "bird"})
esp32["model_split"] = "train"
esp32.loc[esp32["capture_date"].isin(ESP32_VALIDATION_DATES), "model_split"] = "validation"
esp32.loc[esp32["capture_date"].isin(ESP32_TEST_DATES), "model_split"] = "test"
esp32["resolved_path"] = [
    resolve_image_path(stored, relative, ESP32_CAPTURE_ROOT_PATH)
    for stored, relative in zip(esp32["image_path"], esp32["image_relative_path"])
]

bridge = bridge_raw.copy()
bridge["file_exists"] = strict_bool(bridge["file_exists"], "file_exists")
valid_bridge = (
    bridge["file_exists"].fillna(False)
    & bridge["taxonomic_class"].astype(str).str.strip().eq("Aves")
    & bridge["absolute_path"].notna()
    & bridge["absolute_path"].astype(str).str.strip().ne("")
)
rejected_bridge = bridge.loc[~valid_bridge].copy()
bridge = bridge.loc[valid_bridge].copy()

if bridge["bridge_image_id"].duplicated().any():
    raise ValueError("Bridge bridge_image_id values are not unique")
if bridge["absolute_path"].duplicated().any():
    raise ValueError("Bridge absolute_path values are not unique")

bridge["resolved_path"] = [resolve_image_path(path) for path in bridge["absolute_path"]]
bridge["label"] = np.int8(1)
bridge["class_name"] = "bird"
bridge["model_split"] = "train"


def verify_paths(frame, name):
    check = frame if VERIFY_ALL_FILES else frame.sample(min(250, len(frame)), random_state=SEED)
    missing_paths = [path for path in check["resolved_path"] if not Path(path).is_file()]
    if missing_paths:
        raise FileNotFoundError(
            f"{name}: {len(missing_paths)} checked paths are missing; examples: {missing_paths[:5]}"
        )
    print(f"{name}: verified {len(check):,} image paths")


verify_paths(esp32, "ESP32")
verify_paths(bridge, "Bridge")

required_dates = set(ESP32_VALIDATION_DATES) | set(ESP32_TEST_DATES)
missing_dates = required_dates - set(esp32["capture_date"])
if missing_dates:
    raise ValueError(f"Configured holdout dates are absent after motion-crop provenance filtering: {sorted(missing_dates)}")

holdout_start = min(required_dates)
unexpected_dates = sorted(
    date for date in set(esp32["capture_date"]) - required_dates if date >= holdout_start
)
if unexpected_dates:
    raise ValueError(
        "Unconfigured dates at or after the first holdout would leak into training: "
        f"{unexpected_dates}. Assign them explicitly to validation/test or revise the policy."
    )

split_counts = pd.crosstab(esp32["model_split"], esp32["class_name"])
for split in ("train", "validation", "test"):
    if split not in split_counts.index or any(split_counts.loc[split].get(c, 0) == 0 for c in ("bird", "not_bird")):
        raise ValueError(f"ESP32 {split} split must contain both labels after motion-crop provenance filtering")

print(f"ESP32: {len(esp32_raw):,} rows, {len(esp32):,} accepted, {len(rejected_esp32):,} rejected")
print(f"Bridge: {len(bridge_raw):,} rows, {len(bridge):,} accepted, {len(rejected_bridge):,} rejected")
display(rejected_reason_counts.rename("rejected_rows").to_frame())
display(split_counts)
display(pd.crosstab(esp32["capture_date"], esp32["class_name"], margins=True))
display(bridge["source_dataset"].value_counts().rename("bird_images").to_frame())


ValueError: No ESP32 rows have valid matching Pi motion-crop provenance. Rejection counts: {'motion_record_missing': 6381}

## 4. Assemble the binary dataset manifest

All bridge rows are retained as an eligible positive pool, but only 20% of the generated training stream comes from that pool. Validation and test contain ESP32 rows only. The saved dataset manifest carries the normalized ROI coordinates and immutable sidecar provenance for every ESP32 row. Bridge rows explicitly declare `whole_frame_then_letterbox`, with `apply_roi_crop = False`; they never inherit or invent ESP32 provenance.


In [ ]:
esp32_frame = pd.DataFrame({
    "path": esp32["resolved_path"],
    "label": esp32["label"].astype("int8"),
    "class_name": esp32["class_name"],
    "model_split": esp32["model_split"],
    "source_dataset": "esp32_capture",
    "source_role": np.where(esp32["label"].eq(1), "esp32_bird", "esp32_not_bird"),
    "capture_date": esp32["capture_date"],
    "group_id": "esp32_date:" + esp32["capture_date"],
    "original_id": esp32["record_id"].astype(str),
    "crop_policy": ESP32_CROP_POLICY_NAME,
    "apply_roi_crop": True,
    "crop_left": esp32["crop_left"].astype(int),
    "crop_top": esp32["crop_top"].astype(int),
    "crop_right": esp32["crop_right"].astype(int),
    "crop_bottom": esp32["crop_bottom"].astype(int),
    "crop_source": esp32["crop_source"],
    "crop_coordinate_space": "original_decoded_frame_pixels",
    "motion_record_path": esp32["motion_record_path"],
    "motion_record_sha256": esp32["motion_record_sha256"],
    "motion_config_sha256": esp32["motion_config_sha256"],
    
})

bridge_frame = pd.DataFrame({
    "path": bridge["resolved_path"],
    "label": np.int8(1),
    "class_name": "bird",
    "model_split": "train",
    "source_dataset": "bridge_" + bridge["source_dataset"].astype(str),
    "source_role": "bridge_bird",
    "capture_date": "",
    "group_id": "bridge_taxon:" + bridge.get(
        "canonical_taxon_id", pd.Series("unknown", index=bridge.index)
    ).fillna("unknown").astype(str),
    "original_id": bridge["bridge_image_id"].astype(str),
    "crop_policy": BRIDGE_CROP_POLICY_NAME,
    "apply_roi_crop": False,
    # Explicit full-frame coordinates make geometry auditable while apply_roi_crop
    # distinguishes policy from recorded ESP32 ROI provenance.
    "crop_left": 0,
    "crop_top": 0,
    "crop_right": bridge["width_px"].astype(int),
    "crop_bottom": bridge["height_px"].astype(int),
    "crop_source": "bridge_whole_frame",
    "crop_coordinate_space": "original_decoded_frame_pixels",
    "motion_record_path": "",
    "motion_record_sha256": "",
    "motion_config_sha256": "",
})

dataset_manifest = pd.concat([esp32_frame, bridge_frame], ignore_index=True)
required_dataset_columns = [
    "path", "label", "model_split", "source_role", "original_id", "crop_policy",
    "apply_roi_crop", "crop_left", "crop_top", "crop_right", "crop_bottom", "crop_source",
]
if dataset_manifest[required_dataset_columns].isna().any().any():
    raise ValueError("Assembled dataset contains missing required values")

train_esp32_not_bird = esp32_frame.query("model_split == 'train' and label == 0").reset_index(drop=True)
train_esp32_bird = esp32_frame.query("model_split == 'train' and label == 1").reset_index(drop=True)
train_bridge_bird = bridge_frame.reset_index(drop=True)
validation_frame = esp32_frame.query("model_split == 'validation'").reset_index(drop=True)
test_frame = esp32_frame.query("model_split == 'test'").reset_index(drop=True)

for name, frame in {
    "esp32_not_bird": train_esp32_not_bird,
    "esp32_bird": train_esp32_bird,
    "bridge_bird": train_bridge_bird,
}.items():
    if frame.empty:
        raise ValueError(f"Training source {name} is empty")

RUNS_DIR.mkdir(parents=True, exist_ok=True)
created_utc = datetime.now(timezone.utc)
if RUN_NAME is None:
    base_name = f"isbird_manifests_motion_crop_{IMG_SIZE}_{created_utc:%Y%m%d_%H%M%S}"
    candidate = base_name
    suffix = 1
    while (RUNS_DIR / candidate).exists():
        suffix += 1
        candidate = f"{base_name}_{suffix}"
    RUN_NAME = candidate
RUN_DIR = RUNS_DIR / RUN_NAME
RUN_DIR.mkdir(parents=False, exist_ok=False)

DATASET_MANIFEST_PATH = RUN_DIR / "isbird_dataset_manifest.csv"
CROP_REJECTION_REPORT_PATH = RUN_DIR / "esp32_motion_crop_rejections.csv"
dataset_manifest.to_csv(DATASET_MANIFEST_PATH, index=False)
rejected_esp32.to_csv(CROP_REJECTION_REPORT_PATH, index=False)
dataset_manifest_sha = sha256_file(DATASET_MANIFEST_PATH)
crop_rejection_report_sha = sha256_file(CROP_REJECTION_REPORT_PATH)

overview = pd.crosstab(
    [dataset_manifest["model_split"], dataset_manifest["source_dataset"]],
    dataset_manifest["class_name"],
    margins=True,
)
display(overview)
print("Run directory:", RUN_DIR)
print("Dataset manifest SHA-256:", dataset_manifest_sha)
print("Motion crop rejection report:", CROP_REJECTION_REPORT_PATH)
print("Motion crop rejection report SHA-256:", crop_rejection_report_sha)


## 5. Inspect representative inputs


In [ ]:
preview_parts = []
for name, frame in {
    "ESP32 not bird": train_esp32_not_bird,
    "ESP32 bird": train_esp32_bird,
    "bridge bird": train_bridge_bird,
}.items():
    sample = frame.sample(min(3, len(frame)), random_state=SEED).copy()
    sample["preview_group"] = name
    preview_parts.append(sample)
preview = pd.concat(preview_parts, ignore_index=True)

fig, axes = plt.subplots(3, 3, figsize=(12, 9))
for ax, row in zip(axes.flat, preview.itertuples(index=False)):
    image = tf.io.decode_jpeg(tf.io.read_file(row.path), channels=3).numpy()
    ax.imshow(image)
    ax.set_title(f"{row.preview_group}\n{Path(row.path).name}", fontsize=9)
    ax.axis("off")
for ax in axes.flat[len(preview):]:
    ax.axis("off")
plt.tight_layout()


## 6. Per-source crop geometry, augmentation, and `tf.data`

JPEGs are always decoded to RGB, so legacy ESP32 grayscale captures and current RGB captures share the same tensor contract. For ESP32 rows, the normalized acquisition-time ROI is expanded by **20% total in each dimension** (10% per edge), clipped to the original frame, converted to an inclusive-exclusive pixel box with `floor(min)` / `ceil(max)`, and cropped **before** `tf.image.resize_with_pad(..., antialias=True)`. Bridge rows explicitly skip cropping and letterbox the whole frame. Geometry is external to the model and must be reproduced by the receiver; numeric scaling is inside the exported model.


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def resize_geometry(image):
    image = tf.cast(image, tf.float32)
    return tf.image.resize_with_pad(image, IMG_SIZE, IMG_SIZE, antialias=True)


def crop_recorded_selection(image, bounds):
    """Crop already-final half-open original-frame bounds; add no further margin."""
    left, top, right, bottom = tf.unstack(tf.cast(bounds, tf.int32))
    return tf.image.crop_to_bounding_box(image, top, left, bottom - top, right - left)


def load_image(path, bounds, apply_roi_crop):
    encoded = tf.io.read_file(path)
    image = tf.io.decode_jpeg(encoded, channels=3)
    image = tf.cond(
        tf.cast(apply_roi_crop, tf.bool),
        lambda: crop_recorded_selection(image, bounds),
        lambda: image,
    )
    image = resize_geometry(image)
    image = tf.ensure_shape(image, (IMG_SIZE, IMG_SIZE, 3))
    return tf.clip_by_value(image, 0.0, 255.0)


def augment_image(image, index, seed_offset):
    seed = tf.stack([
        tf.cast(SEED + seed_offset, tf.int32),
        tf.cast(tf.math.floormod(index, 2**31 - 1), tf.int32),
    ])
    if AUGMENT_FLIP:
        image = tf.image.stateless_random_flip_left_right(image, seed)
    if AUGMENT_COLOR:
        image = tf.image.stateless_random_brightness(image, 18.0, seed + [0, 1])
        image = tf.image.stateless_random_contrast(image, 0.85, 1.15, seed + [0, 2])
    return tf.clip_by_value(image, 0.0, 255.0)


def frame_tensors(frame):
    bounds = frame[["crop_left", "crop_top", "crop_right", "crop_bottom"]].to_numpy(dtype=np.int32)
    return (
        frame["path"].astype(str).to_numpy(),
        frame["label"].astype("float32").to_numpy(),
        bounds,
        frame["apply_roi_crop"].astype(bool).to_numpy(),
    )


def make_train_component(frame, seed_offset):
    ds = tf.data.Dataset.from_tensor_slices(frame_tensors(frame))
    ds = ds.shuffle(len(frame), seed=SEED + seed_offset, reshuffle_each_iteration=True)
    ds = ds.repeat().enumerate()

    def prepare(index, item):
        path, label, bounds, apply_roi_crop = item
        image = augment_image(load_image(path, bounds, apply_roi_crop), index, seed_offset)
        return image, tf.reshape(tf.cast(label, tf.float32), (1,))

    return ds.map(prepare, num_parallel_calls=AUTOTUNE, deterministic=True)


def make_eval_dataset(frame, batch_size=BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices(frame_tensors(frame))

    def prepare(path, label, bounds, apply_roi_crop):
        return load_image(path, bounds, apply_roi_crop), tf.reshape(label, (1,))

    ds = ds.map(prepare, num_parallel_calls=AUTOTUNE, deterministic=True)
    return ds.batch(batch_size).prefetch(AUTOTUNE)


component_frames = [train_esp32_not_bird, train_esp32_bird, train_bridge_bird]
component_names = ["esp32_not_bird", "esp32_bird", "bridge_bird"]
components = [make_train_component(frame, 100 * (i + 1)) for i, frame in enumerate(component_frames)]
train_stream = tf.data.Dataset.sample_from_datasets(
    components,
    weights=[TRAIN_SOURCE_WEIGHTS[name] for name in component_names],
    seed=SEED,
    stop_on_empty_dataset=False,
)
train_ds = (
    train_stream
    .take(TRAIN_EXAMPLES_PER_EPOCH)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(AUTOTUNE)
)
validation_ds = make_eval_dataset(validation_frame)
test_ds = make_eval_dataset(test_frame)

options = tf.data.Options()
options.experimental_deterministic = True
train_ds = train_ds.with_options(options)
validation_ds = validation_ds.with_options(options)
test_ds = test_ds.with_options(options)

images, labels = next(iter(train_ds))
print("Training batch:", images.shape, images.dtype, "range", float(tf.reduce_min(images)), float(tf.reduce_max(images)))
print("Training labels:", labels.shape, labels.dtype, "positive fraction", float(tf.reduce_mean(labels)))
print("Steps per epoch:", TRAIN_EXAMPLES_PER_EPOCH // BATCH_SIZE)


## 7. MobileNetV2 binary model


In [ ]:
backbone = keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    alpha=BACKBONE_ALPHA,
    include_top=False,
    weights=WEIGHTS,
)
backbone.trainable = False

inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32, name="image")
x = keras.layers.Rescaling(1.0 / 127.5, offset=-1.0, name="scale_to_mobilenet")(inputs)
x = backbone(x, training=False)
x = keras.layers.GlobalAveragePooling2D(name="global_average_pool")(x)
x = keras.layers.Dropout(DROPOUT, name="dropout")(x)
outputs = keras.layers.Dense(1, activation="sigmoid", name="bird_probability")(x)
model = keras.Model(inputs, outputs, name="isbird_mobilenetv2")


def compile_model(target, learning_rate):
    target.compile(
        optimizer=keras.optimizers.Adam(learning_rate),
        loss=keras.losses.BinaryCrossentropy(),
        metrics=[
            keras.metrics.BinaryAccuracy(name="accuracy"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
            keras.metrics.AUC(curve="PR", name="pr_auc"),
            keras.metrics.AUC(curve="ROC", name="roc_auc"),
        ],
    )


compile_model(model, HEAD_LR)
model.summary()


## 8. Train head, then fine-tune

Each stage has an independent best checkpoint. The notebook compares their best validation PR-AUC values and reloads the globally better stage, so a worse fine-tuning stage cannot overwrite a better head-only model.


In [ ]:
def training_callbacks(checkpoint_path, csv_path):
    return [
        keras.callbacks.ModelCheckpoint(
            checkpoint_path, monitor="val_pr_auc", mode="max", save_best_only=True, verbose=1
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_pr_auc", mode="max", patience=EARLY_STOP_PATIENCE, verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_pr_auc", mode="max", factor=0.25,
            patience=max(1, EARLY_STOP_PATIENCE // 2), min_lr=1e-7, verbose=1,
        ),
        keras.callbacks.CSVLogger(csv_path),
        keras.callbacks.TerminateOnNaN(),
    ]


started = time.perf_counter()
stage_records = []

stage1_path = RUN_DIR / "stage1_best.keras"
history_head = model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=HEAD_EPOCHS,
    callbacks=training_callbacks(stage1_path, RUN_DIR / "stage1_history.csv"),
)
stage_records.append({
    "stage": "head",
    "path": stage1_path,
    "best_val_pr_auc": float(np.max(history_head.history["val_pr_auc"])),
})

history_fine = None
if FINE_TUNE_EPOCHS > 0:
    # Fine-tune from the best head-only epoch, not merely the last epoch run.
    model = keras.models.load_model(stage1_path)
    backbone = next(layer for layer in model.layers if layer.name.startswith("mobilenetv2"))
    backbone.trainable = True
    for layer in backbone.layers[:-FINE_TUNE_LAYERS]:
        layer.trainable = False
    for layer in backbone.layers:
        if isinstance(layer, keras.layers.BatchNormalization):
            layer.trainable = False

    compile_model(model, FINE_TUNE_LR)
    stage2_path = RUN_DIR / "stage2_best.keras"
    history_fine = model.fit(
        train_ds,
        validation_data=validation_ds,
        epochs=FINE_TUNE_EPOCHS,
        callbacks=training_callbacks(stage2_path, RUN_DIR / "stage2_history.csv"),
    )
    stage_records.append({
        "stage": "fine_tune",
        "path": stage2_path,
        "best_val_pr_auc": float(np.max(history_fine.history["val_pr_auc"])),
    })

best_stage = max(stage_records, key=lambda item: item["best_val_pr_auc"])
model = keras.models.load_model(best_stage["path"])
train_minutes = (time.perf_counter() - started) / 60.0
print("Selected checkpoint:", best_stage)
print(f"Training time: {train_minutes:.1f} minutes")


In [ ]:
history_rows = []
for stage_name, history_object in (("head", history_head), ("fine_tune", history_fine)):
    if history_object is None:
        continue
    frame = pd.DataFrame(history_object.history)
    frame.insert(0, "epoch", np.arange(1, len(frame) + 1))
    frame.insert(0, "stage", stage_name)
    history_rows.append(frame)
training_history = pd.concat(history_rows, ignore_index=True)
training_history.to_csv(RUN_DIR / "training_history.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, ("loss", "pr_auc", "recall")):
    for stage, frame in training_history.groupby("stage", sort=False):
        ax.plot(frame["epoch"], frame[metric], marker="o", label=f"{stage} train")
        ax.plot(frame["epoch"], frame[f"val_{metric}"], marker="o", linestyle="--", label=f"{stage} validation")
    ax.set_title(metric)
    ax.set_xlabel("stage epoch")
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
plt.tight_layout()


## 9. Save and export TFLite

The representative set is source-stratified across camera negatives, camera birds, and bridge birds. Full-integer conversion uses uint8 input and output tensors; scaling remains embedded in the graph.


In [ ]:
KERAS_PATH = RUN_DIR / "model.keras"
FLOAT_TFLITE_PATH = RUN_DIR / "model_float32.tflite"
UINT8_TFLITE_PATH = RUN_DIR / "model_uint8.tflite"
model.save(KERAS_PATH)

float_converter = tf.lite.TFLiteConverter.from_keras_model(model)
float_tflite = float_converter.convert()
FLOAT_TFLITE_PATH.write_bytes(float_tflite)

per_source = max(1, REPRESENTATIVE_IMAGES // 3)
representative_frame = pd.concat([
    frame.sample(min(per_source, len(frame)), random_state=SEED + i)
    for i, frame in enumerate(component_frames)
], ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)


def representative_dataset():
    for row in representative_frame.itertuples(index=False):
        bounds = np.asarray([row.crop_left, row.crop_top, row.crop_right, row.crop_bottom], dtype=np.int32)
        image = load_image(row.path, bounds, row.apply_roi_crop)
        yield [tf.expand_dims(image, axis=0).numpy().astype(np.float32)]


uint8_converter = tf.lite.TFLiteConverter.from_keras_model(model)
uint8_converter.optimizations = [tf.lite.Optimize.DEFAULT]
uint8_converter.representative_dataset = representative_dataset
uint8_converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
uint8_converter.inference_input_type = tf.uint8
uint8_converter.inference_output_type = tf.uint8
uint8_tflite = uint8_converter.convert()
UINT8_TFLITE_PATH.write_bytes(uint8_tflite)

for path in (KERAS_PATH, FLOAT_TFLITE_PATH, UINT8_TFLITE_PATH):
    print(f"{path.name}: {path.stat().st_size / 1_000_000:.3f} MB | {sha256_file(path)}")


## 10. Inference helpers and exported-model checks


In [ ]:
try:
    from ai_edge_litert.interpreter import Interpreter
    INTERPRETER_RUNTIME = "ai_edge_litert"
except ImportError:
    Interpreter = tf.lite.Interpreter
    INTERPRETER_RUNTIME = "tensorflow"


def selected_eval_frame(frame):
    if TFLITE_EVAL_LIMIT is None or len(frame) <= TFLITE_EVAL_LIMIT:
        return frame.reset_index(drop=True)
    # Keep both classes represented in a deterministic quick evaluation.
    fractions = frame["label"].value_counts(normalize=True)
    parts = []
    for label, group in frame.groupby("label"):
        count = max(1, int(round(TFLITE_EVAL_LIMIT * fractions[label])))
        parts.append(group.sample(min(count, len(group)), random_state=SEED + int(label)))
    return pd.concat(parts).sample(frac=1.0, random_state=SEED).head(TFLITE_EVAL_LIMIT).reset_index(drop=True)


def quantize_input(array, detail):
    dtype = detail["dtype"]
    if not np.issubdtype(dtype, np.integer):
        return array.astype(dtype)
    scale, zero_point = detail["quantization"]
    if scale <= 0:
        raise ValueError("Invalid input quantization scale")
    limits = np.iinfo(dtype)
    return np.clip(np.rint(array / scale + zero_point), limits.min, limits.max).astype(dtype)


def dequantize_output(array, detail):
    if not np.issubdtype(detail["dtype"], np.integer):
        return array.astype(np.float32)
    scale, zero_point = detail["quantization"]
    if scale <= 0:
        raise ValueError("Invalid output quantization scale")
    return (array.astype(np.float32) - zero_point) * scale


def tflite_predict(model_path, frame):
    interpreter = Interpreter(model_path=str(model_path), num_threads=max(1, min(4, os.cpu_count() or 1)))
    interpreter.allocate_tensors()
    input_detail = interpreter.get_input_details()[0]
    output_detail = interpreter.get_output_details()[0]
    probabilities = []
    invoke_seconds = 0.0
    for row in frame.itertuples(index=False):
        bounds = np.asarray([row.crop_left, row.crop_top, row.crop_right, row.crop_bottom], dtype=np.int32)
        array = tf.expand_dims(load_image(row.path, bounds, row.apply_roi_crop), axis=0).numpy()
        interpreter.set_tensor(input_detail["index"], quantize_input(array, input_detail))
        before = time.perf_counter()
        interpreter.invoke()
        invoke_seconds += time.perf_counter() - before
        output = interpreter.get_tensor(output_detail["index"])
        probabilities.append(float(dequantize_output(output, output_detail).reshape(-1)[0]))
    info = {
        "input_shape": input_detail["shape"].tolist(),
        "input_dtype": np.dtype(input_detail["dtype"]).name,
        "input_quantization": list(input_detail["quantization"]),
        "output_dtype": np.dtype(output_detail["dtype"]).name,
        "output_quantization": list(output_detail["quantization"]),
        "mean_invoke_ms": 1000.0 * invoke_seconds / max(1, len(frame)),
    }
    return np.asarray(probabilities, dtype=np.float32), info


def keras_predict(frame):
    return model.predict(make_eval_dataset(frame), verbose=0).reshape(-1).astype(np.float32)


validation_eval = selected_eval_frame(validation_frame)
val_probabilities = {"keras": keras_predict(validation_eval)}
val_probabilities["tflite_float32"], float_info = tflite_predict(FLOAT_TFLITE_PATH, validation_eval)
val_probabilities["tflite_uint8"], uint8_info = tflite_predict(UINT8_TFLITE_PATH, validation_eval)

print("Interpreter runtime:", INTERPRETER_RUNTIME)
print("Float32 tensors:", float_info)
print("Uint8 tensors:", uint8_info)
print("Float32 mean |probability delta| vs Keras:", float(np.mean(np.abs(val_probabilities["keras"] - val_probabilities["tflite_float32"]))))
print("Uint8 mean |probability delta| vs Keras:", float(np.mean(np.abs(val_probabilities["keras"] - val_probabilities["tflite_uint8"]))))


## 11. Select the uint8 validation threshold, then evaluate test once

Threshold candidates are the actual quantized output levels. Selection maximizes validation bird recall while keeping validation false-positive rate at or below the configured limit. Test data is scored only after that choice.


In [ ]:
def safe_ratio(numerator, denominator):
    return float(numerator / denominator) if denominator else np.nan


def binary_metrics(labels, probabilities, threshold):
    labels = np.asarray(labels, dtype=np.int8)
    predicted = np.asarray(probabilities) >= threshold
    positive = labels == 1
    negative = ~positive
    tp = int(np.sum(predicted & positive))
    tn = int(np.sum(~predicted & negative))
    fp = int(np.sum(predicted & negative))
    fn = int(np.sum(~predicted & positive))
    precision = safe_ratio(tp, tp + fp)
    recall = safe_ratio(tp, tp + fn)
    specificity = safe_ratio(tn, tn + fp)
    f1 = 0.0 if precision == 0.0 and recall == 0.0 else (
        safe_ratio(2 * precision * recall, precision + recall)
        if np.isfinite(precision) and np.isfinite(recall) else np.nan
    )
    return {
        "threshold": float(threshold), "images": int(len(labels)),
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
        "accuracy": safe_ratio(tp + tn, len(labels)),
        "balanced_accuracy": np.nanmean([recall, specificity]),
        "precision": precision, "recall": recall, "specificity": specificity,
        "false_positive_rate": safe_ratio(fp, fp + tn),
        "f1": f1,
    }


validation_labels = validation_eval["label"].to_numpy(dtype=np.int8)
quantized_validation = val_probabilities["tflite_uint8"]
candidate_thresholds = np.unique(quantized_validation)
candidate_thresholds = np.append(
    candidate_thresholds,
    np.nextafter(float(candidate_thresholds.max()), np.inf),
)
threshold_table = pd.DataFrame([
    binary_metrics(validation_labels, quantized_validation, threshold)
    for threshold in candidate_thresholds
])
eligible = threshold_table[
    threshold_table["false_positive_rate"] <= MAX_ESP32_VALIDATION_FALSE_POSITIVE_RATE
].copy()
if eligible.empty:
    raise RuntimeError("No threshold satisfies the validation false-positive-rate constraint")

selected = eligible.sort_values(
    ["recall", "f1", "false_positive_rate", "threshold"],
    ascending=[False, False, True, True],
).iloc[0]
SELECTED_THRESHOLD = float(selected["threshold"])
threshold_table.to_csv(RUN_DIR / "validation_thresholds.csv", index=False)
print("Selected uint8 threshold:", SELECTED_THRESHOLD)
display(selected.to_frame("value"))

# The test set is not touched until the validation-only threshold is fixed.
test_eval = selected_eval_frame(test_frame)
test_probabilities = {"keras": keras_predict(test_eval)}
test_probabilities["tflite_float32"], float_test_info = tflite_predict(FLOAT_TFLITE_PATH, test_eval)
test_probabilities["tflite_uint8"], uint8_test_info = tflite_predict(UINT8_TFLITE_PATH, test_eval)


def result_rows(model_name, split_name, frame, probabilities):
    rows = []
    overall = binary_metrics(frame["label"], probabilities, SELECTED_THRESHOLD)
    rows.append({"model": model_name, "split": split_name, "slice": "all", **overall})
    for capture_date, indices in frame.groupby("capture_date").groups.items():
        positions = frame.index.get_indexer(indices)
        metrics = binary_metrics(
            frame.loc[indices, "label"].to_numpy(), probabilities[positions], SELECTED_THRESHOLD
        )
        rows.append({"model": model_name, "split": split_name, "slice": f"date:{capture_date}", **metrics})
    return rows


results = []
for model_name, probabilities in val_probabilities.items():
    results.extend(result_rows(model_name, "validation", validation_eval, probabilities))
for model_name, probabilities in test_probabilities.items():
    results.extend(result_rows(model_name, "test", test_eval, probabilities))
results_table = pd.DataFrame(results)
results_table.to_csv(RUN_DIR / "results.csv", index=False)
display(results_table[results_table["slice"].eq("all")])
display(results_table[results_table["model"].eq("tflite_uint8")])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for split_name, frame, probabilities, color in (
    ("validation", validation_eval, val_probabilities["tflite_uint8"], "tab:blue"),
    ("test", test_eval, test_probabilities["tflite_uint8"], "tab:orange"),
):
    for label, label_name, linestyle in ((0, "not bird", "-"), (1, "bird", "--")):
        mask = frame["label"].to_numpy() == label
        axes[0].hist(
            probabilities[mask], bins=np.linspace(0, 1, 41), histtype="step",
            density=True, color=color, linestyle=linestyle,
            label=f"{split_name}: {label_name}",
        )
axes[0].axvline(SELECTED_THRESHOLD, color="black", linewidth=1.5, label="selected threshold")
axes[0].set(xlabel="uint8 TFLite bird probability", ylabel="density", title="Probability distributions")
axes[0].legend(fontsize=8)

uint8_overall = results_table[
    results_table["model"].eq("tflite_uint8") & results_table["slice"].eq("all")
].set_index("split")
matrix = np.array([
    [uint8_overall.loc["test", "tn"], uint8_overall.loc["test", "fp"]],
    [uint8_overall.loc["test", "fn"], uint8_overall.loc["test", "tp"]],
])
axes[1].imshow(matrix, cmap="Blues")
for row in range(2):
    for column in range(2):
        axes[1].text(column, row, int(matrix[row, column]), ha="center", va="center", fontsize=13)
axes[1].set_xticks([0, 1], ["predicted not bird", "predicted bird"])
axes[1].set_yticks([0, 1], ["actual not bird", "actual bird"])
axes[1].set_title("Uint8 TFLite test confusion matrix")
plt.tight_layout()
plt.savefig(RUN_DIR / "evaluation.png", dpi=160, bbox_inches="tight")


## 12. Save reproducibility metadata and inference contract


In [ ]:
def json_safe(value):
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value


artifact_paths = (KERAS_PATH, FLOAT_TFLITE_PATH, UINT8_TFLITE_PATH)
artifacts = {
    path.name: {
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in artifact_paths
}

uint8_validation_metrics = results_table[
    results_table["model"].eq("tflite_uint8")
    & results_table["split"].eq("validation")
    & results_table["slice"].eq("all")
].iloc[0].to_dict()

output_scale, output_zero_point = uint8_info["output_quantization"]
output_limits = np.iinfo(np.dtype(uint8_info["output_dtype"]))
UINT8_OUTPUT_THRESHOLD = int(np.clip(
    np.ceil(SELECTED_THRESHOLD / output_scale + output_zero_point),
    output_limits.min,
    output_limits.max,
))

threshold_contract = {
    "bird_probability_threshold": SELECTED_THRESHOLD,
    "uint8_output_threshold": UINT8_OUTPUT_THRESHOLD,
    "selected_on": "ESP32 validation rows scored by model_uint8.tflite",
    "selection_policy": "maximize bird recall subject to validation false-positive-rate limit",
    "maximum_validation_false_positive_rate": MAX_ESP32_VALIDATION_FALSE_POSITIVE_RATE,
    "validation_metrics": uint8_validation_metrics,
}
(RUN_DIR / "threshold.json").write_text(
    json.dumps(json_safe(threshold_contract), indent=2, allow_nan=False), encoding="utf-8"
)

inference_contract = {
    "model_file": UINT8_TFLITE_PATH.name,
    "task": "binary bird/not-bird classification",
    "labels": {"0": "not_bird", "1": "bird"},
    "positive_output": "bird_probability",
    "bird_probability_threshold": SELECTED_THRESHOLD,
    "uint8_output_threshold": UINT8_OUTPUT_THRESHOLD,
    "input_shape": [1, IMG_SIZE, IMG_SIZE, 3],
    "input_dtype": uint8_info["input_dtype"],
    "input_quantization": uint8_info["input_quantization"],
    "color_order": "RGB",
    "jpeg_decode_channels": 3,
    "geometry": "selected_crop_then_letterbox",
    "geometry_location": "outside the model",
    "crop_source": ESP32_CROP_SOURCE,
    "crop_coordinate_space": "original_decoded_frame_pixels",
    "crop_bounds_semantics": "half_open_left_top_right_bottom",
    "crop_is_final": True,
    "additional_runtime_margin": 0,
    "source_crop_policy": {
        "esp32_capture": ESP32_CROP_POLICY_NAME,
        "bridge_training_images": BRIDGE_CROP_POLICY_NAME,
    },
    "resize_method": "TensorFlow bilinear",
    "resize_antialias": True,
    "letterbox_padding_value": 0,
    "centering": "TensorFlow resize_with_pad semantics",
    "numeric_scaling": "[0,255] to [-1,1] inside the model",
    "input_quantization_formula": "clip(round(float_pixels / scale + zero_point), dtype_min, dtype_max)",
    "output_dtype": uint8_info["output_dtype"],
    "output_quantization": uint8_info["output_quantization"],
}
(RUN_DIR / "inference_contract.json").write_text(
    json.dumps(json_safe(inference_contract), indent=2, allow_nan=False), encoding="utf-8"
)

run_config = {
    "run_name": RUN_NAME,
    "created_utc": created_utc.isoformat(),
    "task": "binary bird/not-bird classification",
    "quick_run": QUICK_RUN,
    "inputs": {
        "esp32_manifest": str(ESP32_MANIFEST_PATH),
        "esp32_manifest_sha256": sha256_file(ESP32_MANIFEST_PATH),
        "esp32_rows": len(esp32_raw),
        "esp32_accepted_rows": len(esp32),
        "esp32_rejected_rows": len(rejected_esp32),
        "esp32_rejection_reasons": rejected_reason_counts.to_dict(),
        "esp32_motion_crop_rejection_report": str(CROP_REJECTION_REPORT_PATH),
        "esp32_motion_crop_rejection_report_sha256": crop_rejection_report_sha,
        "bridge_manifest": str(BRIDGE_MANIFEST_PATH),
        "bridge_manifest_sha256": sha256_file(BRIDGE_MANIFEST_PATH),
        "bridge_rows": len(bridge_raw),
        "bridge_accepted_rows": len(bridge),
        "dataset_manifest": str(DATASET_MANIFEST_PATH),
        "dataset_manifest_sha256": dataset_manifest_sha,
    },
    "split_policy": {
        "esp32_validation_dates": list(ESP32_VALIDATION_DATES),
        "esp32_test_dates": list(ESP32_TEST_DATES),
        "bridge_images": "training only",
        "esp32_counts": pd.crosstab(esp32_frame["model_split"], esp32_frame["class_name"]).to_dict(),
    },
    "crop_policy": {
        "geometry": "selected_crop_then_letterbox",
        "geometry_location": "outside the model",
        "crop_source": ESP32_CROP_SOURCE,
        "crop_coordinate_space": "original_decoded_frame_pixels",
        "crop_bounds_semantics": "half_open_left_top_right_bottom",
        "crop_is_final": True,
        "additional_runtime_margin": 0,
        "esp32_images": ESP32_CROP_POLICY_NAME,
        "bridge_images": BRIDGE_CROP_POLICY_NAME,
    },
    "model": {
        "preprocessing": "selected_crop_then_letterbox",
        "img_size": IMG_SIZE,
        "backbone": f"MobileNetV2 alpha={BACKBONE_ALPHA} weights={WEIGHTS}",
        "dropout": DROPOUT,
        "selected_checkpoint_stage": best_stage["stage"],
    },
    "training": {
        "batch_size": BATCH_SIZE,
        "examples_per_epoch": TRAIN_EXAMPLES_PER_EPOCH,
        "source_weights": TRAIN_SOURCE_WEIGHTS,
        "head_epochs_requested": HEAD_EPOCHS,
        "fine_tune_epochs_requested": FINE_TUNE_EPOCHS,
        "fine_tune_layers": FINE_TUNE_LAYERS,
        "head_lr": HEAD_LR,
        "fine_tune_lr": FINE_TUNE_LR,
        "seed": SEED,
        "minutes": train_minutes,
    },
    "threshold": threshold_contract,
    "tensorflow": tf.__version__,
    "interpreter_runtime": INTERPRETER_RUNTIME,
    "tflite": {
        "float32_validation": float_info,
        "float32_test": float_test_info,
        "uint8_validation": uint8_info,
        "uint8_test": uint8_test_info,
    },
    "artifacts": artifacts,
}
(RUN_DIR / "run_config.json").write_text(
    json.dumps(json_safe(run_config), indent=2, allow_nan=False), encoding="utf-8"
)

overall = results_table[results_table["slice"].eq("all")]
uint8_test = overall[
    overall["model"].eq("tflite_uint8") & overall["split"].eq("test")
].iloc[0]
log_row = {
    "run_name": RUN_NAME,
    "created_utc": created_utc.isoformat(),
    "quick_run": QUICK_RUN,
    "preprocessing": "selected_crop_then_letterbox",
    "img_size": IMG_SIZE,
    "selected_threshold": SELECTED_THRESHOLD,
    "max_validation_fpr": MAX_ESP32_VALIDATION_FALSE_POSITIVE_RATE,
    "train_minutes": train_minutes,
    "dataset_manifest_sha256": dataset_manifest_sha,
    "uint8_test_accuracy": uint8_test["accuracy"],
    "uint8_test_balanced_accuracy": uint8_test["balanced_accuracy"],
    "uint8_test_precision": uint8_test["precision"],
    "uint8_test_recall": uint8_test["recall"],
    "uint8_test_specificity": uint8_test["specificity"],
    "uint8_test_false_positive_rate": uint8_test["false_positive_rate"],
    "uint8_test_f1": uint8_test["f1"],
    "uint8_size_mb": UINT8_TFLITE_PATH.stat().st_size / 1_000_000,
    "uint8_test_invoke_ms": uint8_test_info["mean_invoke_ms"],
}
log_path = MODELS_DIR / "isbird_manifest_experiments.csv"
new_log_row = pd.DataFrame([log_row])
if log_path.exists():
    old_log = pd.read_csv(log_path)
    if list(old_log.columns) != list(new_log_row.columns):
        raise ValueError(f"Experiment log schema changed: {log_path}")
    experiment_log = pd.concat([old_log, new_log_row], ignore_index=True)
else:
    experiment_log = new_log_row
experiment_log.to_csv(log_path, index=False)

print("Saved complete run to", RUN_DIR)
display(experiment_log.tail())


## 13. Raspberry Pi / receiver inference contract

Use `inference_contract.json` from the run directory as the authority. Its geometry is `selected_crop_then_letterbox`. In outline:

1. Decode JPEG to **RGB** (always three channels).
2. Obtain the already-final pixel crop recorded with that exact capture from `capture_motion_record.selection.final_bounds_px`. The receiver API must receive it explicitly (for example, `infer(image_path, *, roi, roi_source)`); it must not read the camera's current setting or silently use the whole frame when provenance is missing or invalid.
3. Interpret `{left, top, right, bottom}` as half-open original-frame pixel bounds. Crop directly and add **no further margin** because the Pi motion selector already applied context and minimum-size rules.
4. Apply `tf.image.resize_with_pad(..., antialias=True)` semantics to the crop, centered with black padding. (Bridge images were a training-only exception: their explicit policy was `whole_frame_then_letterbox` because no capture ROI provenance exists.)
5. Quantize the `[0, 255]` image using the TFLite input tensor's `(scale, zero_point)`:

   ```python
   quantized = clip(
       round(float_pixels / input_scale + input_zero_point), 0, 255
   ).astype(uint8)
   ```

6. Invoke `model_uint8.tflite`.
7. Prefer the exact integer decision saved as `uint8_output_threshold`:

   ```python
   is_bird = quantized_output >= uint8_output_threshold
   ```

   For reporting a probability, dequantize the scalar output:

   ```python
   bird_probability = (quantized_output - output_zero_point) * output_scale
   ```

Do not apply MobileNet scaling in receiver code: the `[0, 255] → [-1, 1]` transform is already embedded in the model. The saved threshold was chosen against the exported uint8 model, not the pre-conversion Keras model.
